In [2]:
# AI vs ML vs DL vs Generative AI vs Agentic AI

# https://td-mainsite-cdn.tutorialsdojo.com/wp-content/uploads/2025/07/AI-ML-DL.-Gen-AI-Diagram-jpg.webp

In [3]:
# What Generative AI is — what a large language model is actually doing when it answers you

# What is a Large Language Model (LLM)?
# A Large Language Model is an AI model trained on enormous amounts of text and other data to learn patterns in language, relationships between concepts, and how information is commonly expressed.

# At its core, an LLM generates an answer by predicting the next token, one token at a time. A token can be a word, part of a word, or punctuation.

In [4]:
# Tokenization is the process of breaking down text into these smaller units (tokens) that the model can understand and work with. The model uses its training to determine the most likely next token based on the context provided by the previous tokens.
# https://platform.openai.com/tokenizer

In [6]:
import os
from openai import OpenAI
import tiktoken

from dotenv import load_dotenv
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OK Libraries loaded. OpenAI client ready.")

OK Libraries loaded. OpenAI client ready.


## Tokenization

An LLM does not read words. It reads **tokens**.

A token is roughly 3–4 characters in English. Common words are one token. Rare words, proper nouns, and code identifiers are often split into multiple tokens.

**Why this matters for engineering:**
- API costs are charged per token, not per word
- Context window limits are measured in tokens
- Poorly structured prompts waste tokens and increase cost

Let's see exactly how Walmart-relevant text gets tokenized.


In [8]:
# https://developers.openai.com/api/docs/pricing

In [7]:
# Tokenization Demo
enc = tiktoken.encoding_for_model("gpt-4o")

texts = [
    "Walmart",
    "Walmart processes 2.3 million customer transactions per hour.",
    "Supply chain optimization using LangGraph and LLM-as-judge evaluation.",
    "SKU-7829341-B",   # A product identifier — watch how this is split
    "def run_agent(state: AgentState) -> AgentState:",  # Code
]

print(f"Tokenizer: {enc.name}")
print(f"Vocabulary size: {enc.n_vocab:,} tokens")
print()
print(f"{'Text':<55} | {'Tokens':>6} | {'Token IDs (first 8)'}")
print("-" * 90)

for text in texts:
    token_ids = enc.encode(text)
    token_words = [enc.decode([t]) for t in token_ids]
    preview_ids = str(token_ids[:8]) + ("..." if len(token_ids) > 8 else "")
    print(f"{text[:54]:<55} | {len(token_ids):>6} | {preview_ids}")

print()
print("Key insight: 'SKU-7829341-B' is split into multiple tokens.")
print("Structured identifiers cost more tokens than natural text.")


Tokenizer: o200k_base
Vocabulary size: 200,019 tokens

Text                                                    | Tokens | Token IDs (first 8)
------------------------------------------------------------------------------------------
Walmart                                                 |      2 | [54, 40173]
Walmart processes 2.3 million customer transactions pe  |     13 | [54, 40173, 14340, 220, 17, 13, 18, 5749]...
Supply chain optimization using LangGraph and LLM-as-j  |     14 | [49473, 13464, 34658, 2360, 27830, 9922, 326, 451]...
SKU-7829341-B                                           |      6 | [72089, 12, 47574, 47717, 16, 8287]
def run_agent(state: AgentState) -> AgentState:         |     12 | [1314, 2461, 62035, 16114, 25, 28237, 1881, 8]...

Key insight: 'SKU-7829341-B' is split into multiple tokens.
Structured identifiers cost more tokens than natural text.


### Token Cost Calculator

At $0.150 per 1M input tokens (gpt-4o-mini), even small inefficiencies in prompts add up at Walmart's scale.


In [9]:
# Token Cost Calculator
def estimate_cost(text: str, model: str = "gpt-4o-mini", direction: str = "input") -> dict:
    """Estimate the token count and API cost for a given text."""
    enc = tiktoken.encoding_for_model("gpt-4o")  # gpt-4o enc works for gpt-4o-mini too
    token_count = len(enc.encode(text))
    
    # Pricing as of 2026 (approximate, verify at platform.openai.com)
    pricing = {
        "gpt-4o-mini": {"input": 0.150, "output": 0.600},   # per 1M tokens
        "gpt-4o":      {"input": 2.50,  "output": 10.00},
    }
    
    price_per_million = pricing.get(model, {}).get(direction, 0)
    cost_usd = (token_count / 1_000_000) * price_per_million
    
    return {
        "text_chars": len(text),
        "token_count": token_count,
        "chars_per_token": round(len(text) / token_count, 2) if token_count > 0 else 0,
        "cost_usd": round(cost_usd, 8),
        "cost_per_1000_calls": round(cost_usd * 1000, 4),
    }

# Compare prompt efficiency
prompts = {
    "Verbose system prompt": """You are an AI assistant working at Walmart. Your job is to help customer service 
    representatives answer questions. You should always be helpful, professional, and accurate. You must never 
    make up information that you do not know. Always be polite and courteous to customers.""",
    
    "Concise system prompt": """You are a Walmart customer service AI. Be accurate, professional, brief.""",
}

print("Prompt Efficiency Comparison (gpt-4o-mini, input tokens):")
print("-" * 70)
for name, prompt in prompts.items():
    stats = estimate_cost(prompt, "gpt-4o-mini", "input")
    print(f"\n{name}:")
    print(f"  Characters : {stats['text_chars']}")
    print(f"  Tokens     : {stats['token_count']}")
    print(f"  Cost/call  : ${stats['cost_usd']:.6f}")
    print(f"  Cost/1000 calls: ${stats['cost_per_1000_calls']}")

print()
print("At 1M calls/day (Walmart scale), the difference compounds significantly.")


Prompt Efficiency Comparison (gpt-4o-mini, input tokens):
----------------------------------------------------------------------

Verbose system prompt:
  Characters : 284
  Tokens     : 55
  Cost/call  : $0.000008
  Cost/1000 calls: $0.0083

Concise system prompt:
  Characters : 72
  Tokens     : 15
  Cost/call  : $0.000002
  Cost/1000 calls: $0.0023

At 1M calls/day (Walmart scale), the difference compounds significantly.


In [15]:
# Token Cost Calculator
def estimate_cost(text: str, model: str = "gpt-6", direction: str = "input") -> dict:
    """Estimate the token count and API cost for a given text."""
    enc = tiktoken.encoding_for_model("gpt-4o")  # gpt-4o enc works for gpt-4o-mini too
    token_count = len(enc.encode(text))
    
    # Pricing as of 2026 (approximate, verify at platform.openai.com)
    pricing = {
        "gpt-4o-mini": {"input": 0.150, "output": 0.600},   # per 1M tokens
        "gpt-4o":      {"input": 2.50,  "output": 10.00},
        "gpt-6-astra":       {"input": 10.00,  "output": 50.00},  # Hypothetical pricing for gpt-6
    }
    
    price_per_million = pricing.get(model, {}).get(direction, 0)
    cost_usd = (token_count / 1_000_000) * price_per_million
    
    return {
        "text_chars": len(text),
        "token_count": token_count,
        "chars_per_token": round(len(text) / token_count, 2) if token_count > 0 else 0,
        "cost_usd": round(cost_usd, 8),
        "cost_per_million_calls": round(cost_usd * 1000000, 4),
    }

# Compare prompt efficiency
prompts = {
    "Verbose system prompt": """You are an AI assistant working at Walmart. Your job is to help customer service 
    representatives answer questions. You should always be helpful, professional, and accurate. You must never 
    make up information that you do not know. Always be polite and courteous to customers.""",
    
    "Concise system prompt": """You are a Walmart customer service AI. Be accurate, professional, brief.""",
}

print("Prompt Efficiency Comparison (gpt-6-astra, input tokens):")
print("-" * 70)
for name, prompt in prompts.items():
    stats = estimate_cost(prompt, "gpt-6-astra", "input")
    print(f"\n{name}:")
    print(f"  Characters : {stats['text_chars']}")
    print(f"  Tokens     : {stats['token_count']}")
    print(f"  Cost/call  : ${stats['cost_usd']:.6f}")
    print(f"  Cost/1000000 calls: ${stats['cost_per_million_calls']:.4f}")

print()
print("At 1M calls/day (Walmart scale), the difference compounds significantly.")


Prompt Efficiency Comparison (gpt-6-astra, input tokens):
----------------------------------------------------------------------

Verbose system prompt:
  Characters : 284
  Tokens     : 55
  Cost/call  : $0.000550
  Cost/1000000 calls: $550.0000

Concise system prompt:
  Characters : 72
  Tokens     : 15
  Cost/call  : $0.000150
  Cost/1000000 calls: $150.0000

At 1M calls/day (Walmart scale), the difference compounds significantly.
